In [ ]:

import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isEmpty,
    map_cod2name,
    map_name2use_name,
    parallelizeFunction,
    setEF
)
from src.utils.util import (
    loadEstaciones,
    loadEstacionSinCTC
)
from src.processor import (
    XPECProcessor
    
)
from src.utils.topos import getEstacionamientos

In [ ]:
from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
# pd.set_option("future.no_silent_downcasting", True)
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

import argparse
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import yaml
from tqdm.auto import tqdm

from src.api import cargarHistorico, getHistoricoMOW
from src.processor import LogProcessor, xpec_processor
from src.utils import isValidCode, parallelizeFunction, parseDate, rellenarId
from src.api.APIs import (
    getEstadoCirculacionesTecnicas,
    getPlanificacionCirculacionesTecnicas,
    getCirculacionesPlanificadas)

In [ ]:
# Tipos de tren que queremos
train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "SUPRESIÓN",
    "End": "FIN",
    "Entry": "ENTRY",
    "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ORIGEN",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    "TrackingLost": "LOST_TRACK",
}

# Orden lógico de movimientos
mov_sorter = {
    v: k
    for k, v in enumerate(
        [
            "PREVISIÓN",
            "APROXIMACIÓN",
            "EXIT",
            "LLEGADA",
            "FIN",
            "BAJA",
            "ALTA",
            "SALIDA",
            "MANIOBRA_APROXIMACION"
            "MANIOBRA_LLEGADA",
            "MANIOBRA_SALIDA",
        ]
    )
}

In [ ]:
def cargarHistorico(
    start_date: str,
    end_date: str,
    estaciones: list[str],
    trenes: list[str],
    xSIV: bool = True,
    jCTC: bool = False,
    pro: bool = True,
    maniobra:bool = True
):
    # Comprobamos que la fecha de fin sea después de la de inicio
    if end_date <= start_date:
        end_date = (pd.to_datetime(start_date) + timedelta(days=1)).strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    historico = getHistoricoMOW(
        estaciones=estaciones,
        trenes=trenes,
        inicio=start_date,
        fin=end_date,
        xSIV=xSIV,
        jCTC=jCTC,
        pro=pro,
        maniobra= maniobra
    )
    # historico = historico[
    #     (historico["Fecha"] >= pd.to_datetime(start_date))
    #     & (historico["Fecha"] <= pd.to_datetime(end_date))
    # ]
    # Usamos movimientos auditados
    # historico = historico[
    #     np.invert(historico["FuenteVía"].isin(["PLANNED", "SITRA_PROVIDED"]))
    # ]
    historico = historico[historico["NTécnico"].apply(isValidCode)].dropna(
        subset=["Movimiento"]
    )
    historico["mov_ord"] = historico["Movimiento"].apply(mov_sorter.get)
    return historico


In [ ]:
start_date = "2025-10-20"
end_date = "2025-10-21"
estaciones = []

In [ ]:
estaciones = []
# ntrenes = [rellenarId(el) for el in np.arange(100000)]
ntrenes = [rellenarId(el) for el in np.arange(2000, 6000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
planificacion = getCirculacionesPlanificadas("2025-10-20")

In [ ]:
planificacion["Operador"].unique()

In [ ]:
planificacion[planificacion["Operador"] == "L"]

In [ ]:
test = planificacion[planificacion["Código"] == "51003"]

In [ ]:
test2 = test[test["Operador"] =="L"].copy()

In [ ]:
test2

In [ ]:
start_date = "2025-10-20"
end_date = "2025-10-21"
estaciones = []

In [ ]:

estaciones = []
ntrenes = [rellenarId(el) for el in np.arange(100000)]
historico_pre= cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=False,
)
historico_pre = historico_pre.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pre["Día"] = historico_pre["Fecha"].dt.date
historico_pre["day_of_week"] = historico_pre["Fecha"].dt.day_of_week
historico_pre["day_of_year"] = historico_pre["Fecha"].dt.day_of_year
historico_pre["week_of_year"] = (historico_pre["day_of_year"] / 7).astype(int)

In [ ]:
estaciones_sin_ctc = loadEstacionSinCTC()

In [ ]:
estaciones = loadEstaciones()

In [ ]:
import pandas as pd
import yaml
import glob
import os

# Cargar tus datagramas (ejemplo)
# sin_ctc = pd.read_csv("sin_ctc.csv")
# con_ctc = pd.read_csv("con_ctc.csv")

sin_ctc_codigos = set(estaciones_sin_ctc["Código"].astype(str))
con_ctc_codigos = set(estaciones["Código"].astype(str))
dict_dfs = {}

In [ ]:

# Leer todos los archivos YAML de la carpeta
for path in glob.glob("data/Orden movimientos/*.yaml"):
    nombre_archivo = os.path.splitext(os.path.basename(path))[0]
    with open(path, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    registros = []
    for grupo, codigos in data.items():
        for i, codigo in enumerate(codigos, start=1): 
            if codigo in con_ctc_codigos:
                estado = True
            elif codigo in sin_ctc_codigos:
                estado = False
            else:
                estado = "sin registrar"

            registros.append({
                "Línea": grupo,
                "Secuencia": i,
                "Código": codigo,
                "CTC": estado,
            })

    df = pd.DataFrame(registros)
    df["CTC"] = df["CTC"].astype(str)
    dict_dfs[nombre_archivo] = df

In [ ]:
Asturias = dict_dfs["Asturias"]

In [ ]:
coordenada = pd.read_csv("data/coordenada.csv")

In [ ]:
coordenada

In [ ]:
Asturia_coordenada = pd.merge(
    Asturias,
    coordenada,
    on= "Código",
    how = "left"
)

In [ ]:
C1 = Asturia_coordenada[Asturia_coordenada["Línea"]=="C1"]

In [ ]:
C1["Longitud"] = C1["Longitud"].str.replace(",", ".").astype(float)
C1["Latitud"] = C1["Latitud"].str.replace(",", ".").astype(float)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

C1 = C1.sort_values("Secuencia")

# Crear una nueva secuencia con separación uniforme
x_original = C1["Secuencia"]
x = range(len(C1))  # Puntos equidistantes: 0, 1, 2, 3, ...
y = [1]*len(C1)

# Aumentar el tamaño de la figura
plt.figure(figsize=(max(20, len(C1)*1.5), 6))

# Pintar la línea base
plt.plot(x, y, linestyle='-', color='blue', linewidth=2, zorder=1)

# Pintar cada punto según el valor de CTC
for i, (xi, ctc) in enumerate(zip(x, C1['CTC'])):
    if ctc == True or ctc == 'true' or ctc == 'True':
        color = 'green'
    elif ctc == False or ctc == 'false' or ctc == 'False':
        color = 'yellow'
    else:  # No registrado o NaN
        color = 'gray'
    
    plt.plot(xi, 1, marker='o', color=color, markersize=20, zorder=2)

# Etiquetas en su punto correcto sin solapamiento
for i, (xi, codigo, nombre) in enumerate(zip(x, C1['Código'], C1['Nombre'])):
    plt.text(xi, 1.05, f"({codigo})\n{nombre}", 
             ha='center', va='bottom', rotation=45, fontsize=17)

# Crear leyenda
legend_elements = [
    Patch(facecolor='green', label='CTC: True'),
    Patch(facecolor='yellow', label='CTC: False'),
    Patch(facecolor='gray', label='No registrado')
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=15)

plt.title("C1 Asturias", fontsize=14, pad=20)
plt.xlabel("Secuencia de pasos ", fontsize=20)
plt.xticks(x, x_original)  # Mostrar los valores originales de secuencia
plt.yticks([])
plt.ylim(0.8, 1.6)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\trayectoriaC1.png")
plt.savefig(fname,dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
import plotly.graph_objects as go
import pandas as pd

# Ejemplo: carga del datagrama (ajusta el separador si hace falta)
# C1 = pd.read_csv("data/coordenada.csv", sep=";")

# Asegurar que las columnas de coordenadas sean numéricas
C1["Longitud"] = C1["Longitud"].astype(str).str.replace(",", ".").astype(float)
C1["Latitud"] = C1["Latitud"].astype(str).str.replace(",", ".").astype(float)

# Crear etiquetas combinando sólo Código y Nombre (sin coordenadas)
etiquetas = C1["Código"].astype(str) + " - " + C1["Nombre"]

# Asignar colores según el valor de CTC
def asignar_color(ctc):
    if ctc == True or str(ctc).lower() == "true":
        return "green"
    elif ctc == False or str(ctc).lower() == "false":
        return "yellow"
    else:
        return "gray"

colores = C1["CTC"].apply(asignar_color)

# Crear figura
fig = go.Figure(go.Scattermapbox(
    mode="markers+text+lines",
    lon=C1["Longitud"],
    lat=C1["Latitud"],
    text=etiquetas,              
    marker={
        'size': 6,               
        'color': colores
    },
    textposition="top center",
    name="Estaciones"
))

# --- Calcular centro y zoom dinámicos ---
lon_min, lon_max = C1["Longitud"].min(), C1["Longitud"].max()
lat_min, lat_max = C1["Latitud"].min(), C1["Latitud"].max()

center_lon = (lon_min + lon_max) / 2
center_lat = (lat_min + lat_max) / 2

extent = max(lon_max - lon_min, lat_max - lat_min)
if extent < 0.5:
    zoom = 9
elif extent < 1:
    zoom = 8
elif extent < 2:
    zoom = 7
elif extent < 4:
    zoom = 6
else:
    zoom = 5

# Configurar el mapa con estilo más simple
fig.update_layout(
    mapbox={
        'style': "carto-positron",
        'center': {'lon': center_lon, 'lat': center_lat},
        'zoom': zoom
    },
    margin={'l': 0, 't': 0, 'b': 0, 'r': 0},
    title="Trayectoria completa - España"
)

fig.write_html("mapa_trayectoria.html")


In [ ]:
fpath = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Secuencia_estación_ctc.xlsx")

In [ ]:
guardarExcelMulti(dict_dfs,fpath)

In [ ]:
def lista_a_yaml(nombre_grupo: str, texto_codigos: str) -> str:
    """
    Convierte una lista de códigos en texto plano a YAML con el nombre del grupo dado.
    """
    # Limpiar y separar líneas
    codigos = [
        linea.strip().strip('"').strip("'")
        for linea in texto_codigos.splitlines()
        if linea.strip()
    ]

    # Estructura YAML
    estructura = {nombre_grupo: [str(codigo) for codigo in codigos]}

    # Convertir a YAML formateado
    yaml_str = yaml.dump(estructura, allow_unicode=True, sort_keys=False)
    return yaml_str

In [ ]:
text= 

In [ ]:
import json
import requests
from src.api.APIs import hacerPeticion
HOSTPATH = "http://info.api.elcano.operaciones.adif/mse-circulations/msecirculations/planning/day/"
data = {"day": pd.to_datetime(end_date).strftime("%Y-%m-%d")}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    HOSTPATH,
    data=data,
)
res_data = regex.sub(r"\n*data:\s*", ",", response.text)[1:]
res_data = json.loads(f"[{res_data}]")

In [ ]:
with open("resultado.json", "w", encoding="utf-8") as f:
    json.dump(res_data, f, ensure_ascii=False, indent=4)

In [ ]:

def parse_launching_date(ld):
    """
    Convierte launchingDate a pd.Timestamp:
    - Lista/tupla [YYYY, M, D] o [YYYY, M, D, hh, mm, ss]
    - String o cualquier otro formato reconocido por pd.to_datetime
    - Devuelve NaT si no se puede convertir
    """
    try:
        # Caso: lista o tupla
        if isinstance(ld, (list, tuple)) and len(ld) >= 3:
            y, m, d = int(ld[0]), int(ld[1]), int(ld[2])
            if len(ld) >= 6:
                hh, mm, ss = int(ld[3]), int(ld[4]), int(ld[5])
                return pd.Timestamp(year=y, month=m, day=d, hour=hh, minute=mm, second=ss)
            return pd.Timestamp(year=y, month=m, day=d)
        # Caso: otros tipos -> pd.to_datetime intenta convertir
        return pd.to_datetime(ld, errors='coerce')
    except Exception:
        return pd.NaT

In [ ]:
rows = []

for el in res_data:
    cid = el.get("circulationId", {}) or {}
    tecnico = cid.get("number")
    fecha = parse_launching_date(cid.get("launchingDate"))

    day_train = el.get("dayTrain", {}) or {}
    line_dict = day_train.get("line") or {}  # Asegura que sea dict, no None
    line = line_dict.get("name")  # Extrae el nombre de la línea

    company = day_train.get("company")
    operator = day_train.get("operator")
    train_type = day_train.get("trainType")

    steps = day_train.get("journey", {}).get("steps") or []
    for s in steps:
        rows.append({
            "NTécnico": tecnico,
            "FechaOrigen": fecha,
            "Secuencia": s.get("step"),
            "Código": s.get("pointId"),   
            "Vía_Planificada": s.get("parkingTrack"),
            "Línea": line,
            "Compañia": company,
            "Operador": operator,
            "TipoTren": train_type
        })

planificacion = pd.DataFrame(rows)

In [ ]:
estacion_sin_ctc = loadEstacionSinCTC()
estacion_con_ctc = loadEstaciones()
estaciones=pd.read_csv("data/Subdirección_2.csv")

In [ ]:
estaciones

In [ ]:
AV = estaciones[estaciones["Subdirección"] == "RED DE ALTA VELOCIDAD (RAV)"]

In [ ]:
AV

In [ ]:
estacion_completo = pd.concat([estacion_con_ctc,estacion_sin_ctc],ignore_index=True)

In [ ]:
AV["Esta"] = AV["Código"].isin(estacion_completo["Código"])

In [ ]:
AV["Sin_CTC"] = AV["Código"].isin(estacion_sin_ctc["Código"])

In [ ]:
Estación_sin_ctc_AV = AV[AV["Sin_CTC"] == True]
Sin_topo = AV[AV["Esta"]== False]

In [ ]:
# Estación_sin_ctc_AV.drop(columns=["Esta","Sin_CTC"], inplace=True)
Sin_topo.drop(columns=["Esta","Sin_CTC"], inplace=True)

In [ ]:
fname= Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\trenes_por_ambitos.xlsx")

In [ ]:
trenes_por_ambito = pd.read_csv("data/Ámbitos.csv",sep=";")

In [ ]:
guardarExcel(trenes_por_ambito,fname)

In [ ]:
fname = Path(R"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\AV_SIN_CTC.xlsx")
data= {
    "Estaciones_sin_CTC":Estación_sin_ctc_AV,
    "Estación_no_topo":Sin_topo

}
guardarExcelMulti(data,fname)

In [ ]:
sin_ctc = set(estacion_sin_ctc["Código"])
con_ctc = set(estacion_con_ctc["Código"])

In [ ]:
def clasificar_codigo(codigo):
    if codigo in sin_ctc:
        return False
    elif codigo in con_ctc:
        return True
    else:
        return "NO_REGISTRADA"

In [ ]:
planificacion["CTC"] = planificacion["Código"].apply(clasificar_codigo)

In [ ]:
planificacion_rutas = planificacion[["Código", "Secuencia", "CTC", "Ruta", "NTécnico"]].copy()

In [ ]:
def asignar_tecnico(ruta_df):
    if "No_encontrado" in ruta_df["CTC"].values:
        ruta_df["NTécnico"] = ruta_df["NTécnico"].iloc[0]  # o el que corresponda
    else:
        ruta_df["NTécnico"] = None
    return ruta_df

In [ ]:

planificacion_rutas = planificacion_rutas.groupby("Ruta").apply(asignar_tecnico)

In [ ]:
planificacion_rutas_final = planificacion_rutas[["Código", "Secuencia", "CTC", "NTécnico"]]

In [ ]:
planificacion_rutas_final

In [ ]:
planificacion = planificacion.sort_values("Secuencia")

In [ ]:
planificacion["Ruta"] = planificacion.groupby(planificacion.groupby("Secuencia").cumcount())["Código"] \
                                     .transform(lambda x: "_".join(x))

In [ ]:
rutas_unicas = planificacion["Ruta"].unique()

In [ ]:
rutas_unicas

In [ ]:
rutas_dict = {
    ruta: planificacion[planificacion["Ruta"] == ruta].copy()
    for ruta in rutas_unicas
}


In [ ]:
len(rutas_dict)

In [ ]:
archivo_excel = r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\rutas_planificacion.xlsx"

with pd.ExcelWriter(archivo_excel, engine="xlsxwriter") as writer:
    for i, (ruta, df_ruta) in enumerate(rutas_dict.items(), 1):
        nombre_hoja = f"Ruta_{i}"  
        df_ruta.to_excel(writer, sheet_name=nombre_hoja, index=False)

print(f"✅ Todas las rutas guardadas en {archivo_excel}")

<h1> Estación origen sin ctc </h1>

In [ ]:

codigos_seq1 = (
    historico_pro.loc[historico_pro['Secuencia'] == 1, 'Código']
          .dropna()
          .drop_duplicates()
)


In [ ]:

df_est = estaciones_sin_ctc.copy()
df_est['tiene_seq1'] = df_est['Código'].isin(codigos_seq1)


In [ ]:
df_est_seq1 = df_est[df_est['tiene_seq1']].copy()

In [ ]:
df_est_seq1.reset_index(drop=True,inplace=True)

In [ ]:
df_est_seq1.drop(columns=('tiene_seq1'),inplace=True)

In [ ]:
end_date = datetime.now().date().strftime("%Y-%m-%d") 
start_date = datetime.now().date()- timedelta(days=1)
start_date = start_date.strftime("%Y-%m-%d")

In [ ]:
ntrenes = [rellenarId(el) for el in np.arange(100000)]
historico_pro = cargarHistorico(
    start_date,
    end_date,
    estaciones,
    ntrenes,
    xSIV=True,
    jCTC=False,
    pro=True,
)
historico_pro = historico_pro.sort_values(
    by=["FechaOrigen", "NTécnico", "Fecha", "mov_ord"]
).reset_index(drop=True)

# Añadir información de la fecha
historico_pro["Día"] = historico_pro["Fecha"].dt.date
historico_pro["day_of_week"] = historico_pro["Fecha"].dt.day_of_week
historico_pro["day_of_year"] = historico_pro["Fecha"].dt.day_of_year
historico_pro["week_of_year"] = (historico_pro["day_of_year"] / 7).astype(int)

In [ ]:
merge = pd.merge(
    df_est_seq1, 
    historico_pro, 
    how="left", 
    on="Código"
)

In [ ]:
sin_ctc = merge[~merge["Nombre_x"].isna()]
agrupado = sin_ctc.groupby(["Código", "FuenteMovimiento"]).size().reset_index(name="count").sort_values(by="count", ascending=False)

In [ ]:
agrupado.reset_index(drop=True,inplace=True)

In [ ]:
final = agrupado[agrupado["FuenteMovimiento"].isin(["SITRA","CTC_MIE","MSE"])].copy()

In [ ]:
counts = (
final.groupby(["Código", "FuenteMovimiento"])["count"]
.sum()
.unstack(fill_value=0)  # una columna por cada FuenteMovimiento
.reset_index()
)

In [ ]:
final_counts = pd.merge(
    df_est_seq1[["Código", "Nombre"]],
    counts,
    on="Código",
    how="left",
).fillna(0)

In [ ]:
def conteo_sin_ctc (df,df_est_seq1):
    merge = pd.merge(
        df_est_seq1, 
        df, 
        how="left", 
        on="Código"
    )
    sin_ctc = merge[~merge["Nombre_x"].isna()]
    agrupado = sin_ctc.groupby(["Código", "FuenteMovimiento"]).size().reset_index(name="count").sort_values(by="count", ascending=False)
    agrupado.rename(columns={"Nombre_y":"Nombre"}, inplace=True)
    final = agrupado[agrupado["FuenteMovimiento"].isin(["SITRA","CTC_MIE","MSE"])].copy()
    final.sort_values(by=["Código"],inplace=True)
    if final.empty:
        counts = pd.DataFrame(columns=["Código"])  # evitar errores si final está vacío
    else:
        counts = (
        final.groupby(["Código", "FuenteMovimiento"])["count"]
        .sum()
        .unstack(fill_value=0)  # una columna por cada FuenteMovimiento
        .reset_index()
        )
    counts.columns.name = None

    # unir con la lista completa de estaciones (solo Código y Nombre para evitar columnas textuales extras)
    final_counts = pd.merge(
        df_est_seq1[["Código", "Nombre"]],
        counts,
        on="Código",
        how="left",
    ).fillna(0)

    # asegurar tipo entero en los contadores (solo para las columnas que realmente representan conteos)
    for c in final_counts.columns:
        if c not in ("Código", "Nombre"):
        # convertir valores numéricos; si hay valores no convertibles, rellenar con 0
            final_counts[c] = pd.to_numeric(final_counts[c], errors="coerce").fillna(0).astype(int)

    # actualizar final_1 con la tabla resultado para que se use al guardar
    final_2 = final_counts.copy()
    final_3 = pd.merge(
        final_2,
        estaciones_sin_ctc,
        on="Código",
        how="left",
    )
    final_3.drop(columns=["Nombre_y"], inplace=True)
    final_3.rename(columns={"Nombre_x":"Nombre"}, inplace=True)
    return final_3


In [ ]:
df = conteo_sin_ctc(historico_pro,df_est_seq1)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\Elcano Desarrollo\Lidia_informe\estación_sin_ctc_origen.xlsx")

In [ ]:
guardarExcel(df, fname, "Estación_sin_ctc")

<h1>Analisis Chamartin </h1>

In [ ]:
chamartin = historico_pro[historico_pro["Código"] == "17000"].copy()

In [ ]:
solo_ctc = chamartin[
    (chamartin["FuenteVía"] == "CTC") |
    ((chamartin["FuenteVía"] == "UNKNOWN") & (chamartin["Movimiento"] == "FIN"))
]

In [ ]:
completo = historico_pro.copy(
)

In [ ]:
sub_dfs = [group for _, group in solo_ctc.groupby('NTécnico')]

In [ ]:
información_adicional = []

for tren in sub_dfs:
    if isinstance(tren, pd.DataFrame) and 'Movimiento' in tren.columns:
        mask_aproximacion = tren['Movimiento'].str.upper().isin(["APROXIMACIÓN", "PREVISIÓN"])
        mask_llegada = tren['Movimiento'].str.upper() == "LLEGADA"
        mask_origen = tren['Movimiento'].str.upper() == "ORIGEN"
        mask_salida = tren['Movimiento'].str.upper() == "SALIDA"
        mask_finalización = tren['Movimiento'].str.upper() == "FIN"

        aproximacion = tren[mask_aproximacion]
        llegada = tren[mask_llegada]
        origen = tren[mask_origen]
        salida = tren[mask_salida]
        fin = tren[mask_finalización]

        # === Caso ORIGEN ===
        if not origen.empty:
            tiempo_origen = origen["Fecha"].iloc[0]
            NTécnico = origen["NTécnico"].iloc[0]
            Código = origen["Código"].iloc[0]
            Fecha = origen["FechaOrigen"].iloc[0]
            Vía = origen["Vía"].iloc[0] if pd.notna(origen["Vía"].iloc[0]) and origen["Vía"].iloc[0] != "" else "NA"
            Elemento = origen["Elemento"].iloc[0] if pd.notna(origen["Elemento"].iloc[0]) and origen["Elemento"].iloc[0] != "" else "NA"
            Tipo_circulación = "ORIGEN"

            salida_despues = salida[salida["Fecha"] > tiempo_origen]
            if not salida_despues.empty:
                tiempo_salida = salida_despues["Fecha"].iloc[0]
                tiempo_anticipación_CTC = tiempo_salida - tiempo_origen
            else:
                tiempo_anticipación_CTC = "NA"

            if tiempo_anticipación_CTC != "NA":
                horas = tiempo_anticipación_CTC.total_seconds() // 3600
                minutos = (tiempo_anticipación_CTC.total_seconds() % 3600) // 60
                segundos = int(tiempo_anticipación_CTC.total_seconds() % 60)
                tiempo_anticipación_CTC_formateado = f"{int(horas):02}:{int(minutos):02}:{segundos:02}"
            else:
                tiempo_anticipación_CTC_formateado = "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": tiempo_anticipación_CTC_formateado,
                "Tipo circulación": Tipo_circulación
            }
            información_adicional.append(column)

        # === Caso APROXIMACIÓN o PREVISIÓN ===
        elif not aproximacion.empty:
            tiempo_aprox = aproximacion["Fecha"].iloc[0]
            NTécnico = aproximacion["NTécnico"].iloc[0]
            Código = aproximacion["Código"].iloc[0]
            Fecha = aproximacion["FechaOrigen"].iloc[0]
            Vía = aproximacion["Vía"].iloc[0] if pd.notna(aproximacion["Vía"].iloc[0]) and aproximacion["Vía"].iloc[0] != "" else "NA"
            Elemento = aproximacion["Elemento"].iloc[0] if pd.notna(aproximacion["Elemento"].iloc[0]) and aproximacion["Elemento"].iloc[0] != "" else "NA"

            llegada_despues = llegada[llegada["Fecha"] > tiempo_aprox]
            if not llegada_despues.empty:
                tiempo_llegada = llegada_despues["Fecha"].iloc[0]
                tiempo_anticipación_CTC = tiempo_llegada - tiempo_aprox
                Tipo_circulación = "FIN" if not fin.empty else "PASO"
            else:
                tiempo_anticipación_CTC = "NA"
                Tipo_circulación = "INCOMPLETA"

            if tiempo_anticipación_CTC != "NA":
                horas = tiempo_anticipación_CTC.total_seconds() // 3600
                minutos = (tiempo_anticipación_CTC.total_seconds() % 3600) // 60
                segundos = int(tiempo_anticipación_CTC.total_seconds() % 60)
                tiempo_anticipación_CTC_formateado = f"{int(horas):02}:{int(minutos):02}:{segundos:02}"
            else:
                tiempo_anticipación_CTC_formateado = "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": tiempo_anticipación_CTC_formateado,
                "Tipo circulación": Tipo_circulación
            }
            información_adicional.append(column)

        # === Caso SIN ANTICIPACIÓN ===
        elif aproximacion.empty and (
            (not llegada.empty and not salida.empty) or
            (not llegada.empty and not fin.empty)
        ):
            NTécnico = llegada["NTécnico"].iloc[0]
            Código = llegada["Código"].iloc[0]
            Fecha = llegada["FechaOrigen"].iloc[0]
            Vía = llegada["Vía"].iloc[0] if pd.notna(llegada["Vía"].iloc[0]) and llegada["Vía"].iloc[0] != "" else "NA"
            Elemento = llegada["Elemento"].iloc[0] if pd.notna(llegada["Elemento"].iloc[0]) and llegada["Elemento"].iloc[0] != "" else "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": "NA",
                "Tipo circulación": "SIN ANTICIPACIÓN"
            }
            información_adicional.append(column)

        # === Caso DESCONOCIDO ===
        else:
            if not tren.empty:
                NTécnico = tren["NTécnico"].iloc[0]
                Código = tren["Código"].iloc[0]
                Fecha = tren["FechaOrigen"].iloc[0]
                Vía = tren["Vía"].iloc[0] if pd.notna(tren["Vía"].iloc[0]) and tren["Vía"].iloc[0] != "" else "NA"
                Elemento = tren["Elemento"].iloc[0] if pd.notna(tren["Elemento"].iloc[0]) and tren["Elemento"].iloc[0] != "" else "NA"
            else:
                NTécnico = Código = Fecha = Vía = Elemento = "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": "NA",
                "Tipo circulación": "DESCONOCIDA"
            }
            información_adicional.append(column)
    else:
        print("El elemento no es un DataFrame o no tiene columna 'Movimiento'")


In [ ]:

df= pd.DataFrame(información_adicional)

In [ ]:
df.drop(columns="Elemento",inplace=True)

In [ ]:
Paso_fin = df[df["Tipo circulación"].isin(["FIN","PASO"])].copy()

In [ ]:
sub_dfs = [group for _, group in Paso_fin.groupby(["Vía"])]


In [ ]:

tiempos_medios = []

for via_df in sub_dfs:
    codigo = via_df["Código"].iloc[0]
    Via = via_df["Vía"].iloc[0]
    if 'Tiempo de anticipación CTC' in via_df.columns:
        tiempos_validos = pd.to_timedelta(
            via_df["Tiempo de anticipación CTC"], errors="coerce"
        ).dropna()

        if not tiempos_validos.empty:
            tiempo_medio = tiempos_validos.mean()
            # 🔹 Formatear a HH:MM:SS
            total_seconds = int(tiempo_medio.total_seconds())
            horas = total_seconds // 3600
            minutos = (total_seconds % 3600) // 60
            segundos = total_seconds % 60
            tiempo_medio_str = f"{horas:02}:{minutos:02}:{segundos:02}"
        else:
            tiempo_medio_str = "NA"

        tiempos_medios.append({"Código":codigo,"Vía":Via, "Tiempo medio CTC": tiempo_medio_str})

df_tiempos_medios_via = pd.DataFrame(tiempos_medios)


In [ ]:
completo.loc[completo['Secuencia'].notna(), 'Secuencia'] = completo.loc[completo['Secuencia'].notna(), 'Secuencia'].astype(int)


In [ ]:
df["Estación anterior"] = "NA"
df["Código anterior"] = "NA"

for idx, row in df.iterrows():
    ntype = row["Tipo circulación"]

    # Si es ORIGEN, se marca como tal directamente
    if ntype == "ORIGEN":
        df.at[idx, "Estación anterior"] = "ORIGEN"
        df.at[idx, "Código anterior"] = "ORIGEN"
        continue

    # Para otros casos excepto DESCONOCIDA e INCOMPLETA
    if ntype not in ["DESCONOCIDA", "INCOMPLETA"]:
        ntec = row["NTécnico"]
        tren = completo[completo["NTécnico"] == ntec]

        secuencia = tren[tren["Código"] == "17000"]
        if not secuencia.empty:
            secuencia = secuencia.iloc[0]
            N_secuencia = int(secuencia["Secuencia"])
            N_secuencia_anterior = N_secuencia - 1

            # Buscamos la secuencia anterior
            secuencia_anterior = tren[tren["Secuencia"] == N_secuencia_anterior]
            if not secuencia_anterior.empty:
                secuencia_anterior = secuencia_anterior.iloc[0]
                estacion_anterior = secuencia_anterior["Nombre"]
                codigo_anterior = secuencia_anterior["Código"]

                df.at[idx, "Estación anterior"] = estacion_anterior
                df.at[idx, "Código anterior"] = codigo_anterior


In [ ]:
df_paso = df[df["Tipo circulación"].isin(["PASO","FIN"])]

In [ ]:
agrupado = df.groupby(["FechaOrigen","Vía","Elemento","Tipo circulación","Código anterior","Estación anterior"]).size().reset_index(name="Recuento")

In [ ]:
agrupado2 = df.groupby(["FechaOrigen","Elemento","Tipo circulación","Código anterior","Estación anterior"]).size().reset_index(name="Recuento")

In [ ]:
sub_dfs = [group for _, group in df_paso.groupby(["Código anterior", "Estación anterior"])]

tiempos_medios = []

for via_df in sub_dfs:
    codigo = via_df["Código anterior"].iloc[0]
    Estacion = via_df["Estación anterior"].iloc[0]
    # Elemento = via_df["Elemento"].iloc[0]

    if 'Tiempo de anticipación CTC' in via_df.columns:
        tiempos_validos = pd.to_timedelta(
            via_df["Tiempo de anticipación CTC"], errors="coerce"
        ).dropna()

        if not tiempos_validos.empty:
            tiempo_medio = tiempos_validos.mean()
            # 🔹 Formatear a HH:MM:SS
            total_seconds = int(tiempo_medio.total_seconds())
            horas = total_seconds // 3600
            minutos = (total_seconds % 3600) // 60
            segundos = total_seconds % 60
            tiempo_medio_str = f"{horas:02}:{minutos:02}:{segundos:02}"
        else:
            tiempo_medio_str = "NA"

        tiempos_medios.append({"Código":codigo,"Estación":Estacion, "Tiempo medio CTC": tiempo_medio_str})

df_tiempos_medios_nm = pd.DataFrame(tiempos_medios)


In [ ]:
df_tiempos_medios_nm

In [ ]:
chamartin_pre = historico_pre[historico_pre["Código"] == "17000"].copy()

In [ ]:
solo_ctc_pre = chamartin_pre[
    (chamartin_pre["FuenteVía"] == "CTC") |
    ((chamartin_pre["FuenteVía"] == "UNKNOWN") & (chamartin_pre["Movimiento"] == "FIN"))
]

In [ ]:
completo_pre = historico_pre.copy(
)

In [ ]:
sub_dfs = [group for _, group in solo_ctc_pre.groupby('NTécnico')]

In [ ]:
información_adicional = []

for tren in sub_dfs:
    if isinstance(tren, pd.DataFrame) and 'Movimiento' in tren.columns:
        mask_aproximacion = tren['Movimiento'].str.upper().isin(["APROXIMACIÓN", "PREVISIÓN"])
        mask_llegada = tren['Movimiento'].str.upper() == "LLEGADA"
        mask_origen = tren['Movimiento'].str.upper() == "ORIGEN"
        mask_salida = tren['Movimiento'].str.upper() == "SALIDA"
        mask_finalización = tren['Movimiento'].str.upper() == "FIN"

        aproximacion = tren[mask_aproximacion]
        llegada = tren[mask_llegada]
        origen = tren[mask_origen]
        salida = tren[mask_salida]
        fin = tren[mask_finalización]

        # === Caso ORIGEN ===
        if not origen.empty:
            tiempo_origen = origen["Fecha"].iloc[0]
            NTécnico = origen["NTécnico"].iloc[0]
            Código = origen["Código"].iloc[0]
            Fecha = origen["FechaOrigen"].iloc[0]
            Vía = origen["Vía"].iloc[0] if pd.notna(origen["Vía"].iloc[0]) and origen["Vía"].iloc[0] != "" else "NA"
            Elemento = origen["Elemento"].iloc[0] if pd.notna(origen["Elemento"].iloc[0]) and origen["Elemento"].iloc[0] != "" else "NA"
            Tipo_circulación = "ORIGEN"

            salida_despues = salida[salida["Fecha"] > tiempo_origen]
            if not salida_despues.empty:
                tiempo_salida = salida_despues["Fecha"].iloc[0]
                tiempo_anticipación_CTC = tiempo_salida - tiempo_origen
            else:
                tiempo_anticipación_CTC = "NA"

            if tiempo_anticipación_CTC != "NA":
                horas = tiempo_anticipación_CTC.total_seconds() // 3600
                minutos = (tiempo_anticipación_CTC.total_seconds() % 3600) // 60
                segundos = int(tiempo_anticipación_CTC.total_seconds() % 60)
                tiempo_anticipación_CTC_formateado = f"{int(horas):02}:{int(minutos):02}:{segundos:02}"
            else:
                tiempo_anticipación_CTC_formateado = "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": tiempo_anticipación_CTC_formateado,
                "Tipo circulación": Tipo_circulación
            }
            información_adicional.append(column)

        # === Caso APROXIMACIÓN o PREVISIÓN ===
        elif not aproximacion.empty:
            tiempo_aprox = aproximacion["Fecha"].iloc[0]
            NTécnico = aproximacion["NTécnico"].iloc[0]
            Código = aproximacion["Código"].iloc[0]
            Fecha = aproximacion["FechaOrigen"].iloc[0]
            Vía = aproximacion["Vía"].iloc[0] if pd.notna(aproximacion["Vía"].iloc[0]) and aproximacion["Vía"].iloc[0] != "" else "NA"
            Elemento = aproximacion["Elemento"].iloc[0] if pd.notna(aproximacion["Elemento"].iloc[0]) and aproximacion["Elemento"].iloc[0] != "" else "NA"

            llegada_despues = llegada[llegada["Fecha"] > tiempo_aprox]
            if not llegada_despues.empty:
                tiempo_llegada = llegada_despues["Fecha"].iloc[0]
                tiempo_anticipación_CTC = tiempo_llegada - tiempo_aprox
                Tipo_circulación = "FIN" if not fin.empty else "PASO"
            else:
                tiempo_anticipación_CTC = "NA"
                Tipo_circulación = "INCOMPLETA"

            if tiempo_anticipación_CTC != "NA":
                horas = tiempo_anticipación_CTC.total_seconds() // 3600
                minutos = (tiempo_anticipación_CTC.total_seconds() % 3600) // 60
                segundos = int(tiempo_anticipación_CTC.total_seconds() % 60)
                tiempo_anticipación_CTC_formateado = f"{int(horas):02}:{int(minutos):02}:{segundos:02}"
            else:
                tiempo_anticipación_CTC_formateado = "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": tiempo_anticipación_CTC_formateado,
                "Tipo circulación": Tipo_circulación
            }
            información_adicional.append(column)

        # === Caso SIN ANTICIPACIÓN ===
        elif aproximacion.empty and (
            (not llegada.empty and not salida.empty) or
            (not llegada.empty and not fin.empty)
        ):
            NTécnico = llegada["NTécnico"].iloc[0]
            Código = llegada["Código"].iloc[0]
            Fecha = llegada["FechaOrigen"].iloc[0]
            Vía = llegada["Vía"].iloc[0] if pd.notna(llegada["Vía"].iloc[0]) and llegada["Vía"].iloc[0] != "" else "NA"
            Elemento = llegada["Elemento"].iloc[0] if pd.notna(llegada["Elemento"].iloc[0]) and llegada["Elemento"].iloc[0] != "" else "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": "NA",
                "Tipo circulación": "SIN ANTICIPACIÓN"
            }
            información_adicional.append(column)

        # === Caso DESCONOCIDO ===
        else:
            if not tren.empty:
                NTécnico = tren["NTécnico"].iloc[0]
                Código = tren["Código"].iloc[0]
                Fecha = tren["FechaOrigen"].iloc[0]
                Vía = tren["Vía"].iloc[0] if pd.notna(tren["Vía"].iloc[0]) and tren["Vía"].iloc[0] != "" else "NA"
                Elemento = tren["Elemento"].iloc[0] if pd.notna(tren["Elemento"].iloc[0]) and tren["Elemento"].iloc[0] != "" else "NA"
            else:
                NTécnico = Código = Fecha = Vía = Elemento = "NA"

            column = {
                "NTécnico": NTécnico,
                "FechaOrigen": Fecha,
                "Código": Código,
                "Vía": Vía,
                "Elemento": Elemento,
                "Tiempo de anticipación CTC": "NA",
                "Tipo circulación": "DESCONOCIDA"
            }
            información_adicional.append(column)
    else:
        print("El elemento no es un DataFrame o no tiene columna 'Movimiento'")


In [ ]:

df= pd.DataFrame(información_adicional)

In [ ]:
completo_pre.loc[completo_pre['Secuencia'].notna(), 'Secuencia'] = completo_pre.loc[completo_pre['Secuencia'].notna(), 'Secuencia'].astype(int)


In [ ]:
df["Estación anterior"] = "NA"
df["Código anterior"] = "NA"

for idx, row in df.iterrows():
    ntype = row["Tipo circulación"]

    # Si es ORIGEN, se marca como tal directamente
    if ntype == "ORIGEN":
        df.at[idx, "Estación anterior"] = "ORIGEN"
        df.at[idx, "Código anterior"] = "ORIGEN"
        continue

    # Para otros casos excepto DESCONOCIDA e INCOMPLETA
    if ntype not in ["DESCONOCIDA", "INCOMPLETA"]:
        ntec = row["NTécnico"]
        tren = completo_pre[completo_pre["NTécnico"] == ntec]

        secuencia = tren[tren["Código"] == "17000"]
        if not secuencia.empty:
            secuencia = secuencia.iloc[0]
            N_secuencia = int(secuencia["Secuencia"])
            N_secuencia_anterior = N_secuencia - 1

            # Buscamos la secuencia anterior
            secuencia_anterior = tren[tren["Secuencia"] == N_secuencia_anterior]
            if not secuencia_anterior.empty:
                secuencia_anterior = secuencia_anterior.iloc[0]
                estacion_anterior = secuencia_anterior["Nombre"]
                codigo_anterior = secuencia_anterior["Código"]

                df.at[idx, "Estación anterior"] = estacion_anterior
                df.at[idx, "Código anterior"] = codigo_anterior


In [ ]:
df_paso = df[df["Tipo circulación"].isin(["PASO","FIN"])].copy()

In [ ]:
sub_dfs = [group for _, group in df_paso.groupby(["Código anterior", "Estación anterior"])]

tiempos_medios = []

for via_df in sub_dfs:
    codigo = via_df["Código anterior"].iloc[0]
    Estacion = via_df["Estación anterior"].iloc[0]
    # Elemento = via_df["Elemento"].iloc[0]

    if 'Tiempo de anticipación CTC' in via_df.columns:
        tiempos_validos = pd.to_timedelta(
            via_df["Tiempo de anticipación CTC"], errors="coerce"
        ).dropna()

        if not tiempos_validos.empty:
            tiempo_medio = tiempos_validos.mean()
            # 🔹 Formatear a HH:MM:SS
            total_seconds = int(tiempo_medio.total_seconds())
            horas = total_seconds // 3600
            minutos = (total_seconds % 3600) // 60
            segundos = total_seconds % 60
            tiempo_medio_str = f"{horas:02}:{minutos:02}:{segundos:02}"
        else:
            tiempo_medio_str = "NA"

        tiempos_medios.append({"Código":codigo,"Estación":Estacion, "Tiempo medio CTC": tiempo_medio_str})

df_tiempos_medios_nm_pre = pd.DataFrame(tiempos_medios)

In [ ]:
df_tiempos_medios_nm_pre

In [ ]:
fdestino = Path(r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\Elcano Desarrollo\Info SITRA\chamartin.xlsx")

In [ ]:
data ={
    "tiempo_promedio_via":df_tiempos_medios_via,
    "Tiempo_promedio":df_tiempos_medios_nm,
    "Tiempo_promedio_viernes":df_tiempos_medios_nm_pre
}

In [ ]:
guardarExcelMulti(data,fdestino)

In [ ]:
sub_dfs = [group for _, group in df.groupby(["Código anterior", "Estación anterior","Elemento","Vía"])]

tiempos_medios = []

for via_df in sub_dfs:
    via = via_df['Vía'].iloc[0]
    codigo = via_df["Código anterior"].iloc[0]
    Estacion = via_df["Estación anterior"].iloc[0]
    Elemento = via_df["Elemento"].iloc[0]

    if 'Tiempo de anticipación CTC' in via_df.columns:
        tiempos_validos = pd.to_timedelta(
            via_df["Tiempo de anticipación CTC"], errors="coerce"
        ).dropna()

        if not tiempos_validos.empty:
            tiempo_medio = tiempos_validos.mean()
            # 🔹 Formatear a HH:MM:SS
            total_seconds = int(tiempo_medio.total_seconds())
            horas = total_seconds // 3600
            minutos = (total_seconds % 3600) // 60
            segundos = total_seconds % 60
            tiempo_medio_str = f"{horas:02}:{minutos:02}:{segundos:02}"
        else:
            tiempo_medio_str = "NA"

        tiempos_medios.append({"Código":codigo,"Estación":Estacion,"Elemento":Elemento,"Vía": via, "Tiempo medio CTC": tiempo_medio_str})

df_tiempos_medios = pd.DataFrame(tiempos_medios)
print(df_tiempos_medios)


In [ ]:
sub_dfs = [group for _, group in df.groupby(["Código anterior", "Estación anterior","Elemento","Vía","Tipo circulación"])]


In [ ]:
tiempos_medios = []

for via_df in sub_dfs:
    via = via_df['Vía'].iloc[0]
    codigo = via_df["Código anterior"].iloc[0]
    Estacion = via_df["Estación anterior"].iloc[0]
    Elemento = via_df["Elemento"].iloc[0]
    tipo = via_df["Tipo circulación"].iloc[0]

    if 'Tiempo de anticipación CTC' in via_df.columns:
        tiempos_validos = pd.to_timedelta(
            via_df["Tiempo de anticipación CTC"], errors="coerce"
        ).dropna()

        if not tiempos_validos.empty:
            tiempo_medio = tiempos_validos.mean()
            # 🔹 Formatear a HH:MM:SS
            total_seconds = int(tiempo_medio.total_seconds())
            horas = total_seconds // 3600
            minutos = (total_seconds % 3600) // 60
            segundos = total_seconds % 60
            tiempo_medio_str = f"{horas:02}:{minutos:02}:{segundos:02}"
        else:
            tiempo_medio_str = "NA"

        tiempos_medios.append({"Código":codigo,"Estación":Estacion,"Elemento":Elemento,"Vía": via,"Tipo":tipo,"Tiempo medio CTC": tiempo_medio_str})

df_tiempos_medios_tipo = pd.DataFrame(tiempos_medios)
print(df_tiempos_medios)




In [ ]:

df_tiempos_medios_tipo

In [ ]:
agrupado["FechaOrigen"] = pd.to_datetime(agrupado["FechaOrigen"], errors='coerce')
agrupado["FechaOrigen"] = agrupado["FechaOrigen"].dt.strftime("%Y-%m-%d")
df["FechaOrigen"] = pd.to_datetime(df["FechaOrigen"], errors='coerce')
df["FechaOrigen"] = df["FechaOrigen"].dt.strftime("%Y-%m-%d")
agrupado2["FechaOrigen"] = pd.to_datetime(agrupado2["FechaOrigen"], errors='coerce')
agrupado2["FechaOrigen"] = agrupado2["FechaOrigen"].dt.strftime("%Y-%m-%d")

data ={
    "Resumen": df_tiempos_medios_3,
    "Resumen_vía":df_tiempos_medios,
    "Resumen_vía_cir":df_tiempos_medios_tipo,
    "Conteo_sin_vía":agrupado2,
    "Conteo": agrupado,
    "detalle":df,
}
fname = Path(r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\Elcano Desarrollo\Info SITRA\21_09_cha_2.xlsx")
guardarExcelMulti(data,fname)

In [ ]:
def promedio_tiempo(tiempos):

    # Convertir los tiempos a objetos timedelta
    duraciones = []
    for tiempo in tiempos:
        h, m, s = map(int, tiempo.split(":"))
        duraciones.append(timedelta(hours=h, minutes=m, seconds=s))

    # Calcular el promedio
    promedio = sum(duraciones, timedelta()) / len(duraciones)

    # Mostrar el promedio en formato hh:mm:ss
    total_seconds = int(promedio.total_seconds())
    horas = total_seconds // 3600
    minutos = (total_seconds % 3600) // 60
    segundos = total_seconds % 60

    # Formatear el resultado
    promedio_formateado = f"{horas:02}:{minutos:02}:{segundos:02}"
    print("Promedio de tiempo:", promedio_formateado)


In [ ]:
promedio_tiempo(["00:01:21","00:03:03","00:01:53","00:02:08","00:01:46"])

In [ ]:
promedio_tiempo(["00:03:34","00:01:16","00:01:41"])


